In [1]:
import ast
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json
import os
%matplotlib qt

from fovmap.plots_utils import visualize_lens_dir, plot_cone, plot_cone_vert
from fovmap.file_utils import (
    parse_lens_dir_csv_file,
    rotate_vectors,
    load_manual_hex_grid_face_list,
    create_intersect_df
    )
from fovmap.vector_cone_intersect import vector_cone_intersection
from fovmap.surface_plot import (
    complete_cone_surface_apex_neg_z,
    cone_surface_apex_neg_z,
    cone_surface_apex_neg_z_w_blckd_ngl
    )
from fovmap.face_utils import (
    categorize_face,
    plot_hexagonal_surfaces_from_df,
    project_points_on_surface
)
from fovmap.deformation_metrics import (
    calculate_metrics
)
from fovmap.vertices_filtering import filter_vert_z_bound, filter_vert_z_bound_w_none
from fovmap import FILES_PATH, CONFIG_PATH

Get Vertices

In [2]:
dra_d3_csv_path = FILES_PATH / 'ommatidia_dra_d3_labels.json'
with open(dra_d3_csv_path, 'r') as f:
    dra_d3_labels = json.load(f)
str_verts = [x for x in dra_d3_labels.keys()]
labels = [x for x in dra_d3_labels.values()]
dra_d3_labels_df = pd.DataFrame({
    "str_vert": str_verts,
    "label": labels
})
dra_d3_labels_df["vert"] = dra_d3_labels_df["str_vert"].apply(lambda x: ast.literal_eval(x))
dra_d3_labels_df

,str_vert,label,vert
0,"[0.705801676266934, -0.706659624575589, 0.0497...",NA,"[0.705801676266934, -0.706659624575589, 0.0497..."
1,"[0.0185200559645359, -0.954643408411641, -0.29...",NA,"[0.0185200559645359, -0.954643408411641, -0.29..."
2,"[-0.105480720953764, -0.931426688771662, -0.34...",NA,"[-0.105480720953764, -0.931426688771662, -0.34..."
3,"[-0.240224223682285, -0.879075831620144, -0.41...",NA,"[-0.240224223682285, -0.879075831620144, -0.41..."
4,"[-0.402089995997915, -0.780652692099951, -0.47...",NA,"[-0.402089995997915, -0.780652692099951, -0.47..."
...,...,...,...
754,"[0.35511379372401, -0.740356970693075, 0.57075...",D3,"[0.35511379372401, -0.740356970693075, 0.57075..."
755,"[0.399583104120045, -0.687165533402935, 0.6067...",D3,"[0.399583104120045, -0.687165533402935, 0.6067..."
756,"[0.443228257852859, -0.63514441883225, 0.63256...",D3,"[0.443228257852859, -0.63514441883225, 0.63256..."
757,"[0.483790255977314, -0.573179507167302, 0.6613...",D3,"[0.483790255977314, -0.573179507167302, 0.6613..."


In [3]:
dra_d3_vertices = np.array(dra_d3_labels_df[dra_d3_labels_df["label"].isin(["DRA", "D3"])]["vert"].tolist())
dra_vertices = np.array(dra_d3_labels_df[dra_d3_labels_df["label"].isin(["DRA"])]["vert"].tolist())
d3_vertices = np.array(dra_d3_labels_df[dra_d3_labels_df["label"].isin(["D3"])]["vert"].tolist())

In [4]:
viewing_dir = np.array([1, 0, 0])
# Cartesian coordinates rotation
rx = 0
ry = 0
rz = 0
title = f"rx_{rx}_ry_{ry}_rz_{rz}"
# Rotate the lens
if rx or ry or rz:
    rotated_vertices = rotate_vectors(dra_d3_vertices, rx, ry, rz)
    viewing_dir = rotate_vectors(viewing_dir, rx, ry, rz)
    rotated_vertices_dict = {
        ind: dra_d3_vertices[ind].copy() for ind in range(len(dra_d3_vertices))
    }
else:
    rotated_vertices = dra_d3_vertices

In [5]:
visualize_lens_dir(rotated_vertices, viewing_dir)

In [6]:
angles = np.linspace(0, 50, 10)
list_rotated_dra_d3_vertices = []
list_rotated_dra_vertices = []
list_rotated_d3_vertices = []
for angle in angles:
    list_rotated_dra_d3_vertices.append(rotate_vectors(dra_d3_vertices, 0, angle, 0))
    list_rotated_dra_vertices.append(rotate_vectors(dra_vertices, 0, angle, 0))
    list_rotated_d3_vertices.append(rotate_vectors(d3_vertices, 0, angle, 0))
for list_rotated_vert in list_rotated_dra_vertices:
    visualize_lens_dir(list_rotated_vert, viewing_dir)

## Get Cone

In [7]:
cone_config_path = CONFIG_PATH / 'cone_config.json'
with open(cone_config_path, 'r') as f:
    cone_config = json.load(f)

## Cone and Vertices Intersection

In [8]:
pos_height_lim = 100
resolution = 500

In [9]:
real_cone_x, real_cone_y, real_cone_z = cone_surface_apex_neg_z_w_blckd_ngl(
    radius=cone_config['radius'],
    cone_angle_deg=cone_config['cone_angle_deg'],
    angle_revolution_deg=cone_config['angle_revolution_deg'],
    angle_offset_deg=cone_config['angle_offset_deg'],
    height=cone_config['height'],
    height_cutoff=cone_config['height_cutoff'],
    z_base_offset=0,
    resolution=resolution,
    blocked_angle_deg=55,
    outer_blocked_angle_deg=25
)
real_cone_fig, real_cone_ax = plot_cone(real_cone_x, real_cone_y, real_cone_z)

In [10]:
list_intersect_dra_d3_vertices = []
for rotated_verts in list_rotated_dra_d3_vertices:
    intersect_point, ommatidia_to_intersect = vector_cone_intersection(
        radius=cone_config['radius'],
        height=cone_config['height'],
        vertices=rotated_verts
    )
    list_intersect_dra_d3_vertices.append(filter_vert_z_bound(intersect_point.values(), [-cone_config["height_cutoff"], 0]))
list_intersect_dra_vertices = []
for rotated_verts in list_rotated_dra_vertices:
    intersect_point, ommatidia_to_intersect = vector_cone_intersection(
        radius=cone_config['radius'],
        height=cone_config['height'],
        vertices=rotated_verts
    )
    list_intersect_dra_vertices.append(filter_vert_z_bound(intersect_point.values(), [-cone_config["height_cutoff"], 0]))
list_intersect_d3_vertices = []
for rotated_verts in list_rotated_d3_vertices:
    intersect_point, ommatidia_to_intersect = vector_cone_intersection(
        radius=cone_config['radius'],
        height=cone_config['height'],
        vertices=rotated_verts
    )
    list_intersect_d3_vertices.append(filter_vert_z_bound(intersect_point.values(), [-cone_config["height_cutoff"], 0]))

In [11]:
for ind, intersect_vertices in enumerate(list_intersect_dra_vertices):
    if intersect_vertices.tolist():
        plot_cone_vert(real_cone_x, real_cone_y, real_cone_z, intersect_vertices, title=f"Pitch Angle {angles[ind]} deg")

In [17]:
all_intersect_vertices = []
for x in list_intersect_d3_vertices:
    if x.tolist():
        all_intersect_vertices += x.tolist()
plot_cone_vert(real_cone_x, real_cone_y, real_cone_z, np.array(all_intersect_vertices), title=f"All D3 Intersect Vertices\n Pitch Angles {[round(x, 1) for x in angles.tolist()]}")

(<Figure size 640x480 with 1 Axes>,
 <Axes3D: title={'center': 'All D3 Intersect Vertices\n Pitch Angles [0.0, 5.6, 11.1, 16.7, 22.2, 27.8, 33.3, 38.9, 44.4, 50.0]'}, xlabel='X', ylabel='Y', zlabel='Z'>)